In [ ]:
"""
Drowsiness Monitoring System (DMS) on PYNQ

This script integrates OpenCV face detection with a custom FPGA accelerator (MiniVGG).
It performs the following steps:
1. Loads the FPGA bitstream and allocates DMA buffers.
2. Captures video from a USB camera.
3. Detects faces/eyes and stabilizes coordinates.
4. Offloads eye-state classification (Open/Closed) to the FPGA.
5. Triggers an alert if eyes remain closed beyond a threshold.
"""

import numpy as np
import time
import cv2
import os
from pynq import Overlay, allocate
from IPython.display import display, Image, clear_output

# ==========================================
# 1. HARDWARE INITIALIZATION
# ==========================================
print("Loading FPGA Overlay...")
overlay = Overlay("design_1.bit")
dma = overlay.axi_dma_0
vgg = overlay.vgg_accelerator_0

# Start the VGG Accelerator (AP_START)
if vgg.read(0x00) & 0x4: 
    vgg.write(0x00, 0x81) 

# Clean up existing buffers if re-running
if 'input_buffer' in globals(): input_buffer.freebuffer()
if 'output_buffer' in globals(): output_buffer.freebuffer()

# Allocate DMA-ready memory (Non-cached for coherency)
# Input: 32x32 pixels flattened (1024)
# Output: 2 Class scores
input_buffer = allocate(shape=(1024,), dtype=np.uint32)
output_buffer = allocate(shape=(2,), dtype=np.uint32)

print("Hardware Initialized & Buffers Allocated.")

# ==========================================
# 2. CAMERA & MODEL CONFIGURATION
# ==========================================

# Download Haar Cascades if missing
if not os.path.exists('haarcascade_frontalface_default.xml'):
    !wget -q https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml
if not os.path.exists('haarcascade_eye.xml'):
    !wget -q https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_eye.xml

face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier('haarcascade_eye.xml')

def open_camera():
    """Scans for available video devices and returns the first working capture object."""
    for index in range(4):
        print(f"Checking camera at index {index}...")
        cap = cv2.VideoCapture(index, cv2.CAP_V4L2)
        if cap.isOpened():
            ret, _ = cap.read()
            if ret:
                print(f"Success! Camera found at index {index}")
                cap.set(3, 320) # Width
                cap.set(4, 240) # Height
                return cap
            else:
                cap.release()
    raise RuntimeError("No working camera found! Please check USB connection.")

cap = open_camera()
display_handle = display(None, display_id=True)

# Safety Thresholds
CLOSED_THRESHOLD = 5     # Frames before alert triggers
MIN_EYE_SIZE = (20, 20)  # Minimum pixel size for valid detection

# ==========================================
# 3. STABILIZATION UTILITIES
# ==========================================
class BoxStabilizer:
    """Smooths bounding box jitter using an exponential moving average."""
    def __init__(self, persistence=5, alpha=0.7):
        self.box = None
        self.missing_count = 0
        self.persistence = persistence
        self.alpha = alpha

    def update(self, new_box):
        if self.box is None:
            self.box = new_box
            self.missing_count = 0
            return self.int_box(self.box)
        # Apply smoothing: New = (alpha * New) + ((1-alpha) * Old)
        self.box = [ (self.alpha * n) + ((1 - self.alpha) * o) for n, o in zip(new_box, self.box) ]
        self.missing_count = 0
        return self.int_box(self.box)

    def get_box(self):
        """Returns the last known position if detection fails briefly."""
        self.missing_count += 1
        if self.box is not None and self.missing_count <= self.persistence:
            return self.int_box(self.box)
        return None
    
    def int_box(self, box):
        return tuple(map(int, box))

class StateStabilizer:
    """Prevents flickering predictions by requiring consistent states over N frames."""
    def __init__(self, threshold=4):
        self.history = ["Open"] * threshold
        self.threshold = threshold
        self.locked_state = "Open"

    def update(self, new_state):
        self.history.append(new_state)
        if len(self.history) > self.threshold:
            self.history.pop(0)
        
        # Only switch state if the history is unanimous
        if all(s == "Closed" for s in self.history):
            self.locked_state = "Closed"
        elif all(s == "Open" for s in self.history):
            self.locked_state = "Open"
        return self.locked_state

def get_eye_estimation(face_box):
    """Fallback: Guesses eye locations based on face geometry if detection fails."""
    fx, fy, fw, fh = face_box
    # Heuristics for Left and Right eye regions
    lx = int(fx + (fw * 0.15)); ly = int(fy + (fh * 0.25))
    rx = int(fx + (fw * 0.55)); ry = int(fy + (fh * 0.25))
    w = int(fw * 0.30); h = int(fh * 0.30)
    return [(lx, ly, w, h), (rx, ry, w, h)]

# ==========================================
# 4. FPGA INFERENCE ROUTINE
# ==========================================
def run_fpga_inference(eye_img_bgr):
    """
    Offloads image classification to the FPGA via AXI DMA.
    Returns: State ('Open'/'Closed') and Color Code (Green/Red).
    """
    # 1. Preprocessing: Grayscale -> Resize 32x32 -> Flatten
    gray = cv2.cvtColor(eye_img_bgr, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (32, 32))
    
    # 2. Load data into DMA Input Buffer
    input_buffer[:] = resized.flatten().astype(np.uint32)
    input_buffer.flush() 
    
    # 3. Perform DMA Transfer
    dma.sendchannel.transfer(input_buffer)
    dma.recvchannel.transfer(output_buffer)
    
    # 4. Wait for Completion (with Timeout)
    start_time = time.time()
    while not dma.recvchannel.idle:
        if (time.time() - start_time) > 0.1: 
            return "Error", (0, 255, 255) # Timeout (Yellow)
    
    # 5. Read Result
    output_buffer.invalidate()
    score_0 = output_buffer[0] & 0xFF
    score_1 = output_buffer[1] & 0xFF
    
    # 6. Classification Logic
    # If Score 1 > Score 0, the model predicts "Closed"
    if score_1 > score_0:
        return "Closed", (0, 0, 255) # Red
    else:
        return "Open", (0, 255, 0)   # Green

# ==========================================
# 5. MAIN EXECUTION LOOP
# ==========================================
# Initialize Stabilizers
face_stab = BoxStabilizer(persistence=10, alpha=0.5)
left_eye_stab = BoxStabilizer(persistence=15, alpha=0.5) 
right_eye_stab = BoxStabilizer(persistence=15, alpha=0.5)
left_state_stab = StateStabilizer(threshold=4)
right_state_stab = StateStabilizer(threshold=4)

consecutive_closed_frames = 0

print("Starting Monitor (Green=Safe, Red=Danger)...")

try:
    while True:
        # --- Capture Frame ---
        ret, frame = cap.read()
        if not ret:
            time.sleep(0.1)
            continue
        
        # --- Step 1: Face Detection ---
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)
        
        if len(faces) > 0:
            # Pick the largest face and smooth coordinates
            largest_face = sorted(faces, key=lambda f: f[2]*f[3], reverse=True)[0]
            (fx, fy, fw, fh) = face_stab.update(largest_face)
        else:
            # Use last known position
            face_result = face_stab.get_box()
            (fx, fy, fw, fh) = face_result if face_result else (None, None, None, None)

        frame_status = "Open" 
        
        if fx is not None:
            cv2.rectangle(frame, (fx, fy), (fx+fw, fy+fh), (255, 0, 0), 2)
            roi_gray = gray[fy:fy+fh, fx:fx+fw]
            roi_color = frame[fy:fy+fh, fx:fx+fw]
            
            # --- Step 2: Eye Detection ---
            raw_eyes = eye_cascade.detectMultiScale(roi_gray, 1.1, 5, minSize=MIN_EYE_SIZE)
            
            # Filter eyes to lower half of face (reduce false positives)
            valid_eyes = []
            for (ex, ey, ew, eh) in raw_eyes:
                if (ey + eh//2) < (fh // 2):
                    valid_eyes.append((ex, ey, ew, eh))
            valid_eyes.sort(key=lambda e: e[0]) # Sort Left to Right
            
            # --- Step 3: Map Eyes to Stabilizers ---
            final_eyes = []
            
            if len(valid_eyes) >= 2:
                # Best case: Both eyes detected
                l_box = left_eye_stab.update(valid_eyes[0])
                r_box = right_eye_stab.update(valid_eyes[-1])
                final_eyes = [(l_box, 'left'), (r_box, 'right')]
            elif len(valid_eyes) == 0:
                # Worst case: No eyes -> Use history or Geometric Estimation
                l_box = left_eye_stab.get_box()
                r_box = right_eye_stab.get_box()
                
                # If history is empty, guess based on face size
                if l_box is None or r_box is None:
                    g_left, g_right = get_eye_estimation((0, 0, fw, fh))
                    if l_box is None: l_box = left_eye_stab.update(g_left)
                    if r_box is None: r_box = right_eye_stab.update(g_right)
                final_eyes = [(l_box, 'left'), (r_box, 'right')]
            else:
                # Mixed case: One eye detected -> Determine which one based on x-pos
                ex = valid_eyes[0][0]
                g_left, g_right = get_eye_estimation((0, 0, fw, fh))
                if ex < (fw // 2): 
                    l_box = left_eye_stab.update(valid_eyes[0])
                    r_box = right_eye_stab.get_box() or right_eye_stab.update(g_right)
                else: 
                    l_box = left_eye_stab.get_box() or left_eye_stab.update(g_left)
                    r_box = right_eye_stab.update(valid_eyes[0])
                final_eyes = [(l_box, 'left'), (r_box, 'right')]

            # --- Step 4: FPGA Inference & Display ---
            for (box, side) in final_eyes:
                if box is None: continue
                (ex, ey, ew, eh) = box
                
                # Boundary checks
                if ey+eh > fh or ex+ew > fw: continue
                eye_img = roi_color[ey:ey+eh, ex:ex+ew]
                if eye_img.size == 0: continue

                # FPGA Call
                raw_state, color = run_fpga_inference(eye_img)
                
                # Stabilize Result (Prevent flickering)
                if side == 'left':
                    stable_state = left_state_stab.update(raw_state)
                else:
                    stable_state = right_state_stab.update(raw_state)
                
                # Determine Colors
                if stable_state == "Closed":
                    frame_status = "Closed"
                    final_color = (0, 0, 255) # Red
                else:
                    final_color = (0, 255, 0) # Green
                
                cv2.rectangle(roi_color, (ex, ey), (ex+ew, ey+eh), final_color, 2)

        # --- Step 5: Alert Trigger ---
        if frame_status == "Closed":
            consecutive_closed_frames += 1
        else:
            consecutive_closed_frames = 0
            
        if consecutive_closed_frames > CLOSED_THRESHOLD:
            cv2.putText(frame, "DROWSINESS ALERT!", (10, 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        # Update Display
        _, jpeg = cv2.imencode('.jpg', frame)
        display_handle.update(Image(data=jpeg.tobytes()))

except KeyboardInterrupt:
    print("Stopped.")
finally:
    if 'cap' in globals(): cap.release()
    input_buffer.freebuffer()
    output_buffer.freebuffer()
    print("Resources Released.")